In [ ]:
from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import RCP_analysis as rcp
import numpy as np
# import spikeinterface as si

# ------------ USER INPUTS ------------
FILE = Path('/home/bryan/mnt/cullen/Current Project Databases - NHP/2025 Cerebellum prosthesis/Nike/Nike_UA_S1/Blackrock/Nike_UA_S1_002')

REPO_ROOT = Path().resolve().parents[2]
PARAMS    = rcp.load_experiment_params(REPO_ROOT / "config" / "params.yaml", repo_root=REPO_ROOT)

SESSION_LOC = (Path(PARAMS.data_root) / Path(PARAMS.location)).resolve()
OUT_BASE  = SESSION_LOC / "results"; OUT_BASE.mkdir(parents=True, exist_ok=True)
BR_ROOT = SESSION_LOC / "Blackrock"; BR_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT = SESSION_LOC / "Metadata"; METADATA_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_CSV  = METADATA_ROOT / f"{Path(PARAMS.session)}_metadata.csv"
SHIFT_CSV = METADATA_ROOT / "br_to_intan_shifts.csv"

BR_SESSION_FOLDERS = rcp.list_br_sessions(BR_ROOT)

RATES = PARAMS.UA_rate_est
BIN_MS     = float(RATES.get("bin_ms", 1.0))
SIGMA_MS   = float(RATES.get("sigma_ms", 50.0))
THRESH     = float(RATES.get("detect_threshold", 3))
PEAK_SIGN  = str(RATES.get("peak_sign", "both"))

ARTREMV_MS_BEFORE = 5.0
ARTCORR_TAIL_MS   = 5.0
    
XLS = rcp.ua_excel_path(REPO_ROOT, PARAMS.probes)
UA_MAP = rcp.load_UA_mapping_from_excel(XLS) if XLS else None
if UA_MAP is None:
    raise RuntimeError("UA mapping required for mapping on NS6.")

# Sync channels
UA_CFG = PARAMS.probes.get("UA")
CAMERA_SYNC_CH = int(UA_CFG.get("camera_sync_ch", 134))
TRIANGLE_SYNC_CH = int(UA_CFG.get("triangle_sync_ch", 138))

UA_AUX_DATA = OUT_BASE / "aux_data" / "UA"
NPRW_AUX_DATA = OUT_BASE / "aux_data" / "NPRW"
UA_CKPT_OUT = OUT_BASE / "checkpoints" / "UA"; UA_CKPT_OUT.mkdir(parents=True, exist_ok=True)
NPRW_CKPT_ROOT = OUT_BASE / "checkpoints" / "NPRW"

# global_job_kwargs = dict(n_jobs=PARAMS.parallel_jobs, chunk_duration=PARAMS.chunk)
# si.set_global_job_kwargs(**global_job_kwargs)

print(f"=== Session: {FILE.name} ===")
rec_ns6 = se.read_blackrock(FILE, stream_name = 'nsx6', all_annotations=True) # Load neural data

br_idx = int(FILE.name.split('_')[-1]) # resolve br index
n_channels = rec_ns6.get_num_channels()

rec_ns6, idx_rows, ua_elec, ua_nsp, ua_region, ua_region_names, ua_port = rcp.apply_ua_mapping_with_regions(rec_ns6, UA_MAP, br_idx, METADATA_CSV)
UA_probe = ua_region.copy()

rec_hp  = spre.highpass_filter(rec_ns6, freq_min=float(PARAMS.highpass_hz))
region_idx, elec_count = np.unique(UA_probe, return_counts=True)
print(" | ".join(f"{ua_region_names[v]}: {c}" for v, c in zip(region_idx, elec_count)))

=== Session: Nike_UA_S1_002 ===
[MAP] UA port from metadata: 'A'
[MAP] renamed 128/128 rows with UA mapping ('UAe###_NSP###') using port 'A'.
SMA: 30 | Dorsal premotor: 32 | M1 inferior: 31 | M1 superior: 35


In [63]:
ar_npz = np.load('/home/bryan/mnt/cullen/Current Project Databases - NHP/2025 Cerebellum prosthesis/Nike/20251203_NRR_RW011/results/checkpoints/UA/rates__NRR_RW011_005__bin20ms_sigma20ms.npz', allow_pickle = True)

peaks = ar_npz['peaks']
peak_t_ms = ar_npz['peak_t_ms']
ar_npz['counts'].shape[1]

fs = 30000
n_ch = 128
n_seg = 1


sigma_ms = 20
bin_ms = 50
bin_samps = int(bin_ms * 1e-3 * fs)

ch_field, samp_field, seg_field, amp_field = ("channel_index", "sample_index", "segment_index", "amplitude")    
peaks = rcp.python.functions.br_preproc._dedup_peaks(peaks, fs, ch_field=ch_field, samp_field=samp_field,
                    seg_field=seg_field, amp_field=amp_field)
peaks_t_ms = peaks['sample_index']/30

# seg_n_samps, peak_t_ms = _compute_peak_times_per_seg(peaks, recording, fs, n_seg, samp_field, seg_field)

counts_cat, t_cat_ms, blank_mask = rcp.python.functions.br_preproc._bin_counts_and_masks_artrmv(
    peaks, [5233072], n_ch, n_seg, fs, bin_ms, None,
    ch_field, samp_field, seg_field,
)

rate_hz = rcp.python.functions.br_preproc._smooth_counts_with_blanks(counts_cat, blank_mask, sigma_ms, bin_ms)

# rate_hz = ar_npz['rate_hz']



In [62]:
counts_cat.shape

(128, 3489)

/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/core/recording_tools.py:546: UserWarning: chunk_size is greater than the number of samples for segment index 2. Using 2079.
  warnings.warn(


noise_level (no parallelization):   0%|          | 0/80 [00:00<?, ?it/s]

[INFO] Spike bin size: 20.0ms, Average noise level: 25.508721353805903


detect peaks using by_channel_torch (workers: 8 processes):   0%|          | 0/462 [00:00<?, ?it/s]

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

# -------- window (in seconds) --------
T_START_S = 28        # start time of the window (s)
WIN_LEN_S = 2        # length of the window (s)

T_START_MS = T_START_S * 1000.0
T_END_MS   = (T_START_S + WIN_LEN_S) * 1000.0

N_CH_PLOT = 17
ch_idx = np.arange(N_CH_PLOT)

# -------- rate & counts window --------
time_mask = (t_cat_ms >= T_START_MS) & (t_cat_ms <= T_END_MS)
t_plot_s  = t_cat_ms[time_mask] / 1000.0                 # ms -> s in window

rate_plot   = rate_hz[ch_idx][:, time_mask]              # (channels, timebins)
counts_plot = counts_cat[ch_idx][:, time_mask]           # raw bin counts

# -------- spike window --------
chan_field = "channel_index"

spike_mask    = (
    (peaks_t_ms >= T_START_MS) &
    (peaks_t_ms <= T_END_MS) &
    (peaks[chan_field] < N_CH_PLOT)
)
spike_times_s = peaks_t_ms[spike_mask] / 1000.0
spike_chans   = peaks[chan_field][spike_mask]

spikes_per_ch = [spike_times_s[spike_chans == ch] for ch in ch_idx]

# -------- plotting --------
plt.rcParams["font.size"] = 12

fig, (ax_raster, ax_counts, ax_fr) = plt.subplots(
    3, 1,
    figsize=(14, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1, 1]},
)

# --- top: raster ---
ax_raster.eventplot(
    spikes_per_ch,
    lineoffsets=ch_idx,
    linelengths=0.1,
    colors="black",
    linewidths=0.5,
    alpha=1,
)
ax_raster.set_xlim(T_START_S, T_START_S + WIN_LEN_S)
ax_raster.set_ylim(-0.5, N_CH_PLOT - 0.5)
ax_raster.set_yticks([0, 5, 10, 15])
ax_raster.set_ylabel("Channels")
ax_raster.set_title("Raster Plot")
ax_raster.tick_params(axis="x", which="both", labelbottom=False)

# --- middle: bin counts ---
im_counts = ax_counts.imshow(
    counts_plot,
    aspect="auto",
    origin="lower",
    extent=[t_plot_s[0], t_plot_s[-1], -0.5, N_CH_PLOT - 0.5],
    cmap="jet",
)
ax_counts.set_xlim(T_START_S, T_START_S + WIN_LEN_S)
ax_counts.set_yticks([0, 5, 10, 15])
ax_counts.set_ylabel("Channel")
ax_counts.set_title("Bin Counts")
ax_counts.tick_params(axis="x", which="both", labelbottom=False)
# optional colorbar:
fig.colorbar(im_counts, ax=ax_counts, label="Counts / bin")

# --- bottom: firing-rate heatmap ---
im_fr = ax_fr.imshow(
    rate_plot,
    aspect="auto",
    origin="lower",
    extent=[t_plot_s[0], t_plot_s[-1], -0.5, N_CH_PLOT - 0.5],
    cmap="jet",
)
ax_fr.set_xlim(T_START_S, T_START_S + WIN_LEN_S)
ax_fr.set_yticks([0, 5, 10, 15])
ax_fr.set_ylabel("Channel")
ax_fr.set_xlabel("Time (s)")
ax_fr.set_title("Firing Rate (spk/sec)")
# optional colorbar:
fig.colorbar(im_fr, ax=ax_fr, label="Firing rate (Hz)")

fig.tight_layout()
fig.subplots_adjust(hspace=0.35)
plt.show()


NameError: name 't_cat_ms' is not defined

In [2]:
# ---------------------------------------------------------------------
# Three UA-region heatmaps in ONE figure with custom height ratios
# + M1i/M1s boundary + electrode-range labels + impedance masking
# ---------------------------------------------------------------------
import numpy as np
import h5py, re, csv
import matplotlib.pyplot as plt
from pathlib import Path

# -------- window (in seconds) --------
T_START_S = 10.0      # start time of the window (s)
WIN_LEN_S = 20.0      # length of the window (s)

with METADATA_CSV.open("r", newline="") as f:
    rdr = csv.DictReader(f)
    if not rdr.fieldnames:
        print(f"[WARN] {METADATA_CSV.name} has no header")
    elif "BR_File" not in rdr.fieldnames or "UA_port" not in rdr.fieldnames:
        print(f"[WARN] metadata missing BR_File and/or UA_port columns (have: {rdr.fieldnames})")
    else:
        for row in rdr:
            try:
                if int(row["BR_File"]) == br_idx:
                    ua_port = (row["UA_port"] or "") or None
                    break
            except Exception:
                continue
            
T_START_MS = T_START_S * 1000.0
T_END_MS   = (T_START_S + WIN_LEN_S) * 1000.0

IMP_BASE = OUT_BASE.parents[1] / "Nike_UA_S1"
IMP_FILES = {
    "A": IMP_BASE / "Impedances" / "bank_A_imp",
    "B": IMP_BASE / "Impedances" / "bank_B_imp",
}

UA_IMP_MAX_KOHM = 1000.0
EXCLUDE_UA_HIGH_Z = True

_imp_pat_elecnum = re.compile(
    r"\belec\s*\d+\s*-\s*(\d{1,3})\s+"          # elecX-### column
    r"(?:<=\s*|≤\s*)?"                          # optional '<=' OR Unicode '≤'
    r"([0-9]+(?:\.[0-9]+)?)\s*"                 # numeric value (e.g. 15, 15.0)
    r"(k?ohms?|kΩ|ohms?|Ω)\b",                  # unit: kOhm, Ohm, kΩ, Ω
    flags=re.IGNORECASE,
)


def _unit_to_kohm(val: float, unit: str) -> float:
    u = (unit or "").lower()
    if "k" in u:
        return float(val)
    return float(val) / 1000.0  # Ω → kΩ

def _read_text_loose(p: Path) -> str:
    if not p.exists():
        raise FileNotFoundError(str(p))
    b = p.read_bytes()
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1", "mac_roman"):
        try:
            return b.decode(enc)
        except Exception:
            pass
    filtered = bytes(ch for ch in b if 9 <= ch <= 126 or ch in (10, 13))
    return filtered.decode("latin-1", errors="ignore")

def load_impedances_from_textedit_dump(path_like: str | Path) -> dict[int, float]:
    """
    Parse lines like 'elec1-5   201 kOhm' from a loose text dump.
    Returns { Elec#: impedance_kΩ }.
    """
    txt = _read_text_loose(Path(path_like))
    out: dict[int, float] = {}
    for m in _imp_pat_elecnum.finditer(txt):
        elec_str, val_str, unit = m.groups()
        try:
            elec = int(elec_str)
            val = float(val_str)
        except Exception:
            continue
        out[elec] = _unit_to_kohm(val, unit)
    return out

def make_ua_keep_mask_from_imp(ua_ids_1based, imp_by_elec, max_kohm):
    """
    Build a boolean mask of UA rows to keep.

    Policy:
      - If an electrode has a measured impedance <= max_kohm, keep it.
      - If impedance is > max_kohm, drop it.
      - If there is NO impedance entry for that electrode, drop it.
    """
    ids_arr = np.asarray(ua_ids_1based, float).ravel()
    keep = np.ones(ids_arr.size, dtype=bool)

    n_matched = 0
    n_highZ   = 0
    n_missing = 0

    for r, eid in enumerate(ids_arr):
        if not np.isfinite(eid):
            # no usable electrode id → drop
            keep[r] = False
            n_missing += 1
            continue

        z = imp_by_elec.get(int(eid))
        if z is None or not np.isfinite(z):
            # no impedance entry → drop
            keep[r] = False
            n_missing += 1
            continue

        n_matched += 1
        if z > max_kohm:
            keep[r] = False
            n_highZ += 1

    print(
        f"[imp mask] matched={n_matched}, "
        f"highZ(>{max_kohm} kΩ)={n_highZ}, "
        f"missing_or_nan={n_missing}, "
        f"kept={keep.sum()}/{keep.size}"
    )

    if not keep.any():
        print("[warn] UA: impedance mask removed all rows; skipping mask.")
        return None

    return keep

# --- build impedance dict once (reuse same IMP_FILES + loader) ---
imp_by_elec = {}

p = IMP_FILES.get(ua_port)
if not p:
    print(f"[warn] UA impedance path missing for port {ua_port} in IMP_FILES.")
else:
    if not p.exists():
        print(f"[warn] UA impedance file not found for port {ua_port}: {p}")
    else:
        try:
            d = load_impedances_from_textedit_dump(p)
            imp_by_elec.update(d)
            print(f"[info] Parsed {len(d)} UA impedances from {p.name} (port {ua_port}).")
        except Exception as e:
            print(f"[warn] could not parse UA impedances from {p} (port {ua_port}): {e}")


# --- apply impedance mask on FULL UA arrays first -------------------
if EXCLUDE_UA_HIGH_Z and imp_by_elec:
    ua_keep_mask = make_ua_keep_mask_from_imp(ua_elec, imp_by_elec, UA_IMP_MAX_KOHM)
    if ua_keep_mask is not None:
        ua_keep_mask = np.asarray(ua_keep_mask, bool)
        rate_hz  = rate_hz[ua_keep_mask, :]   # <--- full time axis
        UA_probe = np.asarray(UA_probe)[ua_keep_mask]
        ua_elec  = np.asarray(ua_elec)[ua_keep_mask]
        print(f"[info] UA long window: kept {ua_keep_mask.sum()}/{ua_keep_mask.size} rows (≤ {UA_IMP_MAX_KOHM:g} kΩ).")

# -------- now pick time window (AFTER masking channels) -------------
time_mask = (t_cat_ms >= T_START_MS) & (t_cat_ms <= T_END_MS)
if not np.any(time_mask):
    print(f"No bins in requested window [{T_START_S}, {T_START_S + WIN_LEN_S}] s")
else:
    t_plot_ms = t_cat_ms[time_mask]
    t_plot_s  = t_plot_ms / 1000.0
    rate_win  = rate_hz[:, time_mask]

    ua_regions  = np.asarray(UA_probe, int)
    ua_elec_arr = np.asarray(ua_elec)   # 1-based electrode IDs, same order as UA_probe

    if ua_regions.size != rate_win.shape[0]:
        print("[WARN] UA_probe length != #channels in rate_win; cannot split by region.")
    elif ua_elec_arr.size != rate_win.shape[0]:
        print("[WARN] ua_elec length != #channels in rate_win; cannot use electrode labels.")
        ua_elec_arr = None
    else:
        # global color range
        vmin_global = 0.0
        if np.isfinite(rate_win).any():
            vmax_global = np.nanpercentile(rate_win, 99.0)
        else:
            vmax_global = 1.0

        # region codes: 0=SMA, 1=PMd, 2=M1 inf, 3=M1 sup
        region_groups = [
            ("M1i+M1s", [2, 3]),
            ("PMd",     [1]),
            ("SMA",     [0]),
        ]

        # ---- build matrices for each group first ----
        group_mats = []  # list of (label, rate_g, regs_g, elec_g)
        for group_label, codes in region_groups:
            mask = np.isin(ua_regions, codes)
            n_ch = int(mask.sum())
            if n_ch == 0:
                print(f"[INFO] No channels in group {group_label}; skipping.")
                continue

            rate_g = rate_win[mask, :]
            regs_g = ua_regions[mask]
            elec_g = ua_elec_arr[mask] if ua_elec_arr is not None else None

            # stable sort within group by region code then electrode ID
            if elec_g is not None:
                sort_order = np.lexsort((elec_g, regs_g))
            else:
                sort_order = np.argsort(regs_g, kind="stable")

            rate_g = rate_g[sort_order, :]
            regs_g = regs_g[sort_order]
            elec_g = elec_g[sort_order] if elec_g is not None else None

            group_mats.append((group_label, rate_g, regs_g, elec_g))

        if not group_mats:
            print("[INFO] No UA channels in any group; nothing to plot.")
        else:
            n_panels = len(group_mats)

            # ---- choose height ratios (one per UA panel) ----
            height_ratios = []
            for label, _, _, _ in group_mats:
                if label == "M1i+M1s":
                    height_ratios.append(2.0)   # taller
                elif label == "PMd":
                    height_ratios.append(1.0)
                elif label == "SMA":
                    height_ratios.append(1.0)
                else:
                    height_ratios.append(1.0)

            # ---- add one more row at bottom for RSW trace ----
            height_ratios.append(0.8)   # relative height for RSW panel

            fig, axes = plt.subplots(
                n_panels + 1, 1,
                figsize=(10, 2 * (n_panels + 1)),
                sharex=True,
                gridspec_kw={"height_ratios": height_ratios}
            )

            axes = list(np.atleast_1d(axes))
            ua_axes = axes[:-1]
            ax_RSW  = axes[-1]

            ua_im = None  # will hold the last UA imshow for colorbar

            for ax, (group_label, rate_g, regs_g, elec_g) in zip(ua_axes, group_mats):
                n_ch = rate_g.shape[0]

                ua_im = ax.imshow(
                    rate_g,
                    aspect="auto",
                    origin="lower",
                    extent=[t_plot_s[0], t_plot_s[-1], 0, n_ch],
                    cmap="jet",
                    vmin=vmin_global,
                    vmax=vmax_global,
                )

            axes = list(np.atleast_1d(axes))
            ua_axes = axes[:-1]
            ax_RSW  = axes[-1]

            ua_im = None  # will hold the last UA imshow for colorbar

            for ax, (group_label, rate_g, regs_g, elec_g) in zip(ua_axes, group_mats):
                n_ch = rate_g.shape[0]

                ua_im = ax.imshow(
                    rate_g,
                    aspect="auto",
                    origin="lower",
                    extent=[t_plot_s[0], t_plot_s[-1], 0, n_ch],
                    cmap="jet",
                    vmin=vmin_global,
                    vmax=vmax_global,
                )

                # ---- y-axis tick labels in electrode ID space ----
                if elec_g is not None and elec_g.size == n_ch:
                    elec_min = int(np.nanmin(elec_g))
                    elec_max = int(np.nanmax(elec_g))

                    n_ticks = min(5, n_ch)
                    yticks  = np.linspace(0.5, n_ch - 0.5, n_ticks)
                    ylabels = np.linspace(elec_min, elec_max, n_ticks).astype(int)

                    ax.set_yticks(yticks)
                    ax.set_yticklabels(ylabels)
                else:
                    ax.set_yticks([])

                ax.set_xlim(T_START_S, T_START_S + WIN_LEN_S)
                ax.set_xlabel("")  # UA panels share x with bottom panel

                # ----- y-label with nominal channel/electrode range -----
                if group_label == "SMA":
                    ylab = "SMA (1–64)"
                elif group_label == "PMd":
                    ylab = "PMd (65–128)"
                elif group_label == "M1i+M1s":
                    ylab = "M1i (129–256) M1s"
                else:
                    ylab = group_label
                ax.set_ylabel(ylab)

                # titles: only the first UA panel shows FILE.name
                if ax is ua_axes[0]:
                    ax.set_title(f"{FILE.name} ({group_label}, nCh={n_ch})")
                else:
                    ax.set_title(f"{group_label} (nCh={n_ch})")

                # horizontal separators between region codes in this group
                unique_regs, reg_start_idx = np.unique(regs_g, return_index=True)
                for y in reg_start_idx[1:]:
                    ax.axhline(y, color="white", linewidth=0.8, alpha=0.7)

                # --- gray line between M1 inferior & M1 superior inside M1i+M1s ---
                if group_label == "M1i+M1s":
                    is_inf = (regs_g == 2)   # 2 = M1i
                    inf_count = int(np.count_nonzero(is_inf))
                    if 0 < inf_count < n_ch:
                        ax.axhline(
                            inf_count,
                            color="0.5",
                            linewidth=1.2,
                            alpha=0.9
                        )

                cbar = plt.colorbar(ua_im, ax=ax)
                cbar.set_label("Firing rate (Hz)")

            # ---------- Bottom panel: RSW trace ----------
            rsw_path = IMP_BASE / "Nike_UA_S1_002_BR.mat"
            fs_RSW = 1000  # Hz
            with h5py.File(rsw_path, "r") as f:
                RSW = f["Data"]["RSW"][0, :]

            t_rsw = np.arange(RSW.size) / fs_RSW
            T_LIMIT_S_START = T_START_S
            T_LIMIT_S       = T_START_S + WIN_LEN_S

            ax_RSW.plot(t_rsw, RSW)
            ax_RSW.set_xlim(T_LIMIT_S_START, T_LIMIT_S)
            ax_RSW.set_ylabel("RSW")
            ax_RSW.set_xlabel("Time (s)")
            ax_RSW.set_title("RSW trace")

            # --- add an *invisible* colorbar to RSW axis so its width matches UA axes ---
            if ua_im is not None:
                cb_dummy = plt.colorbar(ua_im, ax=ax_RSW)
                cb_dummy.ax.set_visible(False)   # keep the layout effect, hide the bar

            plt.tight_layout()
            plt.show()

[info] Parsed 128 UA impedances from bank_A_imp (port A).
[imp mask] matched=128, highZ(>1000.0 kΩ)=2, missing_or_nan=0, kept=126/128


NameError: name 'rate_hz' is not defined